In [ ]:
!sudo apt-get update
!sudo apt-get install -y libavfilter-dev libavformat-dev libavcodec-dev libswscale-dev libavdevice-dev ffmpeg

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchcodec
!pip install av bitsandbytes sentencepiece accelerate opencv-python-headless torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0

## 程式碼說明

### 1. **卸載現有套件**
- 使用 `pip uninstall` 指令移除當前安裝的 transformers 套件
- `-y` 標誌用於自動確認卸載操作，避免出現確認提示

### 2. **安裝開發版本**
- 從 Hugging Face 的官方 GitHub 倉庫安裝最新開發版本
- 這種安裝方式確保獲得對 Gemma 4 模型的最新支援
- 開發版本通常包含最新的功能和錯誤修復

### 注意事項
- 開發版本可能不如穩定版本穩定
- 建議在虛擬環境中進行此操作
- 可能需要重新啟動 kernel 才能使新安裝的套件生效

In [ ]:
# 卸載當前安裝的 transformers 套件
# -y 參數表示自動確認卸載，不需要手動確認
!pip uninstall -y transformers

# 2. 安裝支援 Gemma 4 的最新開發版本
# 從 Hugging Face 的 GitHub 倉庫直接安裝最新開發版 transformers
# git+https:// 表示從 Git 倉庫安裝套件
!pip install git+https://github.com/huggingface/transformers.git

## 程式碼說明

### 1. **導入必要庫**
- `torch`: PyTorch深度學習框架
- `transformers`: Hugging Face的transformers庫，提供預訓練模型

### 2. **模型配置**
- 使用Google的Gemma-4-E2B-IT模型，這是一個多模態語言模型
- 支持處理文字和圖像等多種類型的輸入

### 3. **數據類型選擇**
- `torch.bfloat16`：腦浮點16位，在保持數值穩定性的同時減少記憶體使用

### 4. **設備映射**
- `device_map="auto"`：自動檢測並使用可用的GPU，如果沒有GPU則使用CPU

### 應用場景
這個設置適合用於多模態任務，如：
- 圖像描述
- 視覺問答
- 多模態對話系統

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

# 1. 設置模型和處理器
model_id = "google/gemma-4-e2b-it"  # 指定要使用的模型名稱

# 從預訓練模型加載處理器，用於處理多模態輸入（文字、圖像等）
processor = AutoProcessor.from_pretrained(model_id)

# 從預訓練模型加載多模態語言模型
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,  # 模型識別碼
    torch_dtype=torch.bfloat16,  # 使用bfloat16數據類型以減少記憶體使用並加速計算
    device_map="auto"  # 自動將模型分配到可用的設備（GPU/CPU）
)

## 程式碼說明

### 1. **訊息結構**
- 使用字典格式定義多模態輸入
- `role`: 指定對話參與者角色（user/assistant）
- `content`: 包含多種類型的內容（視頻、文字）

### 2. **處理器參數說明**
- `add_generation_prompt=True`: 在輸入末尾添加特殊標記，提示模型開始生成
- `tokenize=True`: 將文字轉換為模型可理解的數字表示
- `return_dict=True`: 返回結構化的字典對象
- `return_tensors="pt"`: 輸出PyTorch張量格式

### 3. **設備轉移**
- `.to(model.device)`: 確保輸入數據與模型位於相同的計算設備上

### 功能說明
這段代碼準備多模態輸入（視頻+文字），並將其轉換為模型可處理的格式，為後續的推理生成做準備。

In [ ]:
messages = [
    {
        "role": "user",  # 訊息發送者角色為用戶
        "content": [
            {"type": "video", "video": "bottle-detection.mp4"},  # 視頻內容，指定視頻文件路徑
            {"type": "text", "text": "What is happening in this video? List any objects you see."}  # 文字內容，提出問題
        ]
    }
]

# 3. 處理輸入並生成回應
inputs = processor.apply_chat_template(
    messages,  # 傳入對話訊息列表
    add_generation_prompt=True,  # 添加生成提示，指示模型開始生成回應
    tokenize=True,  # 將輸入轉換為token（詞元）
    return_dict=True,  # 以字典形式返回結果
    return_tensors="pt"  # 返回PyTorch張量格式
).to(model.device)  # 將輸入數據移動到模型所在的設備（GPU/CPU）

## 程式碼說明

### 1. **模型生成階段**
- `model.generate()`: 調用模型的生成方法產生回應
- `max_new_tokens=256`: 控制生成內容的長度，避免過長的輸出

### 2. **結果解碼**
- `outputs[0]`: 取得批次中的第一個生成結果
- `inputs["input_ids"].shape[-1]`: 計算輸入序列的長度，用於區分輸入和生成部分
- 切片操作 `[start:]`: 從輸入結束位置開始截取，只保留模型生成的新內容

### 3. **特殊token處理**
- `skip_special_tokens=True`: 過濾掉模型使用的特殊標記，使輸出更易讀

### 功能說明
這段代碼執行模型推理生成回應，並將生成的token序列解碼為人類可讀的文字格式，同時過濾掉不必要的技術性標記。

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=256)
# 使用模型生成回應
# **inputs: 將inputs字典展開為關鍵字參數傳遞給generate方法
# max_new_tokens=256: 限制模型生成的最大新token數量為256個

# 4. 解碼生成結果
# 跳過輸入token，只顯示模型的回應部分
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
# outputs[0]: 取得第一個批次的生成結果
# inputs["input_ids"].shape[-1]: 獲取輸入token序列的長度
# [inputs["input_ids"].shape[-1]:]: 切片操作，跳過輸入部分，只取模型生成的部分
# skip_special_tokens=True: 跳過特殊token（如開始/結束標記），只顯示有意義的文字內容